Let's build the deck!

In [3]:
from itertools import combinations
import random as rd

ranks = '23456789TJQKA'
suits = 'cdhs'

players = 3

rank_values = {}
for i in range(2,15):
    rank_values[ranks[i-2]] = i

hand_ranks = {
    "High card": 0,
    "Pair": 1,
    "Two pair": 2,
    "Three of a kind": 3,
    "Straight": 4,
    "Flush": 5,
    "Full house": 6,
    "Four of a kind": 7,
    "Straight flush": 8,
    "Royal flush": 9
}

def create_deck():
    deck = []

    for rank in ranks:
        for suit in suits:
            deck.append((rank,suit))
    
    return deck

def deal(players, deck):
    '''
    keep player amount between 2 and 20 players
    '''
    rd.shuffle(deck)

    hands = []

    for i in range(players):
        hand = []

        for j in range(2):
            hand.append(deck.pop()) # removes last element 
    
        hands.append(hand)
    
    return hands


def deal_board(deck):
    flop = [deck.pop() for _ in range(3)]
    turn = deck.pop()
    river = deck.pop()

    return flop + [turn, river]

deck = create_deck()

hands = deal(players, deck)
board = deal_board(deck) 

def evaluate_hands(hands, board):
    
    all_hand_types = []
    all_scores = []

    for hand in hands:
        cards = hand + board

        rank_counts = {}
        suit_counts = {}
    
        for card in cards:
            rank = card[0]
            suit = card[1]

            if rank in rank_counts:
                rank_counts[rank] += 1
            else:
                rank_counts[rank] = 1

            if suit in suit_counts:
                suit_counts[suit] += 1
            else:
                suit_counts[suit] = 1

        pairs = [x for x in rank_counts if rank_counts[x] == 2]
        num_pairs = len(pairs)

        trips = [x for x in rank_counts if rank_counts[x] == 3]
        num_trips = len(trips)

        quads = [x for x in rank_counts if rank_counts[x] == 4]
        num_quads = len(quads)

        # check for flush
        flush = False
        flush_suit = ''

        for n in suit_counts:
            if suit_counts[n] >= 5:
                flush = True
                flush_suit = n
                break

        # check for straight
        rank_numbers = [rank_values[card[0]] for card in cards]
        unique_ranks = sorted(set(rank_numbers))

        if 14 in unique_ranks:
            unique_ranks.insert(0,1)

        straight = False

        for n in range(len(unique_ranks) - 4): 
            if unique_ranks[n+4] - unique_ranks[n] == 4:
                straight = True
                straight_high = unique_ranks[n+4]
                break

        # check for straight flush
        straight_flush = False
        royal_flush = False

        if flush:
            flush_cards = [card for card in cards if card[1] == flush_suit]

            flush_rank_numbers = [rank_values[card[0]] for card in flush_cards]
            unique_flush_ranks = sorted(set(flush_rank_numbers))

            if 14 in unique_flush_ranks:
                unique_flush_ranks.insert(0,1)
            
            for i in range(len(unique_flush_ranks) - 4):
                if unique_flush_ranks[i+4] - unique_flush_ranks[i] == 4:
                    straight_flush = True
                    straight_flush_high = unique_flush_ranks[i+4]
                    break

            if all(rank in flush_rank_numbers for rank in [10, 11, 12, 13, 14]):
                royal_flush = True

        if royal_flush:
            hand_type = 'Royal flush'

            score = (9,)

        elif straight_flush:
            hand_type = "Straight flush"

            score = (8, straight_flush_high)

        elif num_quads == 1:
            hand_type = "Four of a kind"

            kicker = max(
                [rank_values[card[0]] for card in cards 
                if card[0] not in quads]
            )
            
            score = (7, rank_values[quads[0]], kicker)
        
        elif num_trips >= 2:
            hand_type = 'Full house'

            trip_ranks = sorted(
                [rank_values[x] for x in trips],
                reverse=True)

            score = (6, trip_ranks[0], trip_ranks[1])

        elif num_trips == 1 and num_pairs == 1:
            hand_type = "Full house"

            trips_rank = rank_values[trips[0]]
            pairs_rank = max(
                [rank_values[card[0]]
                for card in cards 
                if card[0] in pairs])

            score = (6, trips_rank, pairs_rank)

        elif flush:
            hand_type = 'Flush'

            flush_cards = sorted(
                [rank_values[card[0]] 
                for card in cards 
                if card[1] == flush_suit],
                reverse=True)
            
            score = (5, *flush_cards[:5])

            
        
        elif straight:
            hand_type = 'Straight'

            score = (4, straight_high)

        elif num_trips == 1:
            hand_type = "Three of a kind"

            trip_rank = rank_values[trips[0]]

            kickers = sorted([rank_values[card[0]]
                            for card in cards 
                            if card[0] not in trips]
                            , reverse=True)
            
            score = (3, trip_rank, *kickers[:2])

        elif num_pairs >= 2:
            hand_type = "Two pair"

            pairs_ranks = sorted([rank_values[x] for x in pairs], reverse=True)[:2]
            kicker = max(
                rank_values[card[0]] 
                for card in cards 
                if card[0] not in pairs)

            score = (2, *pairs_ranks, kicker)


        elif num_pairs == 1:
            hand_type = "Pair"
            pair_rank = rank_values[pairs[0]]

            kickers = [
                rank_values[card[0]]
                for card in cards
                if card[0] != pairs[0]
            ]

            kickers = sorted(kickers, reverse=True)

            score = (1, pair_rank, *kickers[:3])

        else:
            hand_type = "High card"

            sorted_ranks = sorted(rank_numbers, reverse=True)

            score = (0, *sorted_ranks[:5])

        all_hand_types.append(hand_type)
        all_scores.append(score)
        
    return hands, board, all_hand_types, all_scores


def determine_winner(all_scores):
    highest = max(all_scores)
    winners = []

    for i in range(len(all_scores)):
        if all_scores[i] == highest:
            winners.append(i+1)
    
    return winners


def random_game():
    deck = create_deck()

    hands = deal(players, deck)
    board = deal_board(deck) 

    return evaluate_hands(hands, board)


In [4]:

def remaining_deck(known_cards):
    deck = create_deck()

    for card in known_cards:
        deck.remove(card)

    return deck

def poker_equity(hero, villains, board):
    all_hands = [hero] + villains

    known_cards = board.copy()

    for hand in all_hands:
        known_cards += hand

    cards_needed = 5 - len(board)

    deck = remaining_deck(known_cards)
    runouts = list(combinations(deck, cards_needed))

    hero_wins = 0
    hero_share = 0

    for runout in runouts:
        new_board = board + list(runout)
        
        _, _, all_hand_types, all_scores = evaluate_hands(all_hands, new_board)

        winners = determine_winner(all_scores)

        if 1 in winners:
            if len(winners) == 1:
                hero_wins += 1
            else:
                hero_share += 1 / len(winners)

    equity = (hero_wins + hero_share) / len(runouts)

    return equity


In [5]:
def monte_carlo_equity(hero, villains, board, simulations):
    all_hands = [hero] + villains

    hero_wins = 0
    hero_share = 0

    for _ in range(simulations):

        known_cards = board.copy()

        for hand in all_hands:
            known_cards += hand

        deck = remaining_deck(known_cards)
        rd.shuffle(deck)

        cards_needed = 5 - len(board)

        runout = deck[:cards_needed]

        new_board = board + runout

        _, _, all_hand_types, all_scores = evaluate_hands(all_hands, new_board)

        winners = determine_winner(all_scores)

        if 1 in winners:
            if len(winners) == 1:
                hero_wins += 1
            else:
                hero_share += 1/len(winners)

    equity = (hero_wins + hero_share) / simulations

    return equity
    
hero = [('A', 's'), ('A', 'c')]

villains = [
    [('Q', 'h'), ('Q', 'd')],
    [('J', 'c'), ('J', 'd')]
]

board = [('8', 's'), ('3', 's'), ('9', 'c')]

equity = monte_carlo_equity(hero, villains, board, 10000)

print("monte carlo hero equity:", equity*100, '%')

equity = poker_equity(hero, villains, board)

print("exact hero equity:", equity*100, '%')

monte carlo hero equity: 83.61 %
exact hero equity: 82.94573643410853 %


In [13]:
def generate_pair(hand):
    suits = 'cdsh'

    cards = []

    for suit in suits:
        cards.append((hand[0], suit))

    pairs = [list(pair) for pair in combinations(cards, 2)]

    return pairs

def generate_suited(rank1, rank2):
    suits = 'cdhs'

    hands = []

    for suit in suits:
        hands.append([(rank1, suit), (rank2, suit)])

    return hands

def generate_offsuit(rank1, rank2):
    suits = 'cdsh'

    hands = []

    for suit1 in suits:
        for suit2 in suits:
            
            if suit1 != suit2:
                hands.append([(rank1, suit1), (rank2, suit2)])
    
    return hands

def generate_hands(hand):
    if hand[0] == hand[1]:
        return generate_pair(hand[0])

    elif hand[2] == 's':
        return generate_suited(hand[0], hand[1])

    elif hand[2] == 'o':
        return generate_offsuit(hand[0], hand[1])


def generate_range(range_list, known_cards):

    all_hands = []

    for hand in range_list:
        hands = generate_hands(hand)

        hands = remove_known_cards(hands, known_cards)    

        all_hands += hands

    return all_hands

def generate_weighted_range(weighted_range, known_cards):

    all_hands = []

    for hand, weight in weighted_range.items():
        hands = generate_hands(hand)

        hands = remove_known_cards(hands, known_cards)

        for combination in hands:
            all_hands.append((combination, weight))

    return all_hands
                   

In [8]:

def range_equity(hero, villain_range, board):
    total_equity = 0
    
    for villain_hand in villain_range:
        equity = poker_equity(hero, [villain_hand], board)

        total_equity += equity

    
    equity = total_equity / len(villain_range)

    return equity

def weighted_range_equity(hero, weighted_hands, board):
    total_equity = 0
    total_weight = 0

    for villain_hand, weight in weighted_hands:
        equity = poker_equity(hero, [villain_hand], board)

        total_equity += equity * weight
        total_weight += weight

    return total_equity / total_weight
    


In [14]:
def remove_known_cards(hands, known_cards):

    valid_hands = []

    for hand in hands:

        if all(card not in known_cards for card in hand):
            valid_hands.append(hand)

    return valid_hands

hero = [('A', 's'), ('A', 'c')]

board = [('8', 's'), ('3', 's'), ('9', 'c')]

known_cards = hero + board

villain_range = ["QQ", "JJ", "TT", "AKs", "AQs"]
villain_hands = generate_range(villain_range, known_cards)

equity = range_equity(hero, villain_hands, board)

for hand in villain_range:
    hands = generate_hands(hand)
    hands = remove_known_cards(hands, known_cards)

    print(hand, ":", len(hands), "combinations")
print(equity * 100, '%')
print(villain_hands)
print("Number of combinations:", len(villain_hands))

QQ : 6 combinations
JJ : 6 combinations
TT : 6 combinations
AKs : 2 combinations
AQs : 2 combinations
90.79889807162532 %
[[('Q', 'c'), ('Q', 'd')], [('Q', 'c'), ('Q', 's')], [('Q', 'c'), ('Q', 'h')], [('Q', 'd'), ('Q', 's')], [('Q', 'd'), ('Q', 'h')], [('Q', 's'), ('Q', 'h')], [('J', 'c'), ('J', 'd')], [('J', 'c'), ('J', 's')], [('J', 'c'), ('J', 'h')], [('J', 'd'), ('J', 's')], [('J', 'd'), ('J', 'h')], [('J', 's'), ('J', 'h')], [('T', 'c'), ('T', 'd')], [('T', 'c'), ('T', 's')], [('T', 'c'), ('T', 'h')], [('T', 'd'), ('T', 's')], [('T', 'd'), ('T', 'h')], [('T', 's'), ('T', 'h')], [('A', 'd'), ('K', 'd')], [('A', 'h'), ('K', 'h')], [('A', 'd'), ('Q', 'd')], [('A', 'h'), ('Q', 'h')]]
Number of combinations: 22


In [15]:
known_cards = hero + board

weighted_range = {
    "QQ": 1.0,
    "JJ": 1.0,
    "TT": 0.5,
    "AKs": 0.5,
    "AQs": 0.25
}

weighted_hands = generate_weighted_range(
    weighted_range,
    known_cards
)

equity = weighted_range_equity(
    hero,
    weighted_hands,
    board
)

print("Hero equity:", equity * 100, "%")

Hero equity: 90.19283746556474 %


In [19]:


def calc_pot_odds(pot, bet, call):
    pot_after_bet = pot + bet
    pot_after_call = pot_after_bet + call

    return call/pot_after_call

def break_even_equity(pot, bet, call):
    pot_after_bet = pot + bet

    return call / pot_after_bet

pot = 100
bet = 25
call = 25

pot_odds = calc_pot_odds(pot, bet, call)

print(pot_odds)

def ev_dollars(equity, pot, bet, call):
    pot_after_bet = pot + bet
    
    ev = equity * pot_after_bet - call

    return ev
print(ev_dollars(equity, pot, bet, call))



0.16666666666666666
87.74104683195591
